# Proyecto Centinela - Fase 1: Centinela de la Papa
### Redes Neuronales - Deep Learning (EA-N-F-004) - Maestria en Ciencia de Datos, USTA
Integrantes: Juan Guillermo Montano Junca y Jair Andres Valderrama Cano - version v1

Linea base con un MLP sobre la imagen aplanada a tabla (rasgos de color y textura por hoja) para predecir **hoja enferma: si/no** (tizon temprano o tardio frente a sana). Datos: Irish Potato Imagery Dataset (Zenodo, CC BY 4.0) + clima de NASA POWER (entra en la Fase 2).

> Metodologia "Ingenieria de Trinchera": si no corre en Colab, no existe. El cuaderno corre con Ejecutar todo de principio a fin.

## 0. Preparacion del entorno

In [ ]:
# Instalar/paquetes (en Colab ya vienen casi todos)
# !pip install torch pillow scikit-learn matplotlib pandas -q
import io, zipfile, os, random
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_recall_curve

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 1. El problema y los datos

Cada foto de hoja se resume en un vector de rasgos tabulares (promedios y desviaciones por canal R, G, B; exceso de verde; fraccion de pixeles parduzcos; brillo; textura). Esos rasgos son la "version aplanada" de la imagen que entrena el MLP; en la Fase 2 se reemplazan por una CNN.

**Rutas:** sube los tres .zip de Zenodo a Colab (o monta Drive) y ajusta `ZIPS`. Para iterar rapido usa `POR_CLASE` (p. ej. 1500); pon `None` para el conjunto completo.

In [ ]:
# Descarga (opcional) directo de Zenodo en Colab:
# !wget -q -O healthy.zip  "https://zenodo.org/records/8286529/files/healthy.zip?download=1"
# !wget -q -O earlyblt.zip "https://zenodo.org/records/8286529/files/earlyblt.zip?download=1"
# !wget -q -O lateblt.zip  "https://zenodo.org/records/8286529/files/lateblt.zip?download=1"

ZIPS = {   # clase -> (ruta del zip, etiqueta 0=sana 1=enferma)
    "sana":     ("healthy.zip",  0),
    "temprano": ("earlyblt.zip", 1),
    "tardio":   ("lateblt.zip",  1),
}
POR_CLASE = 1500        # fotos por clase para iterar; None = todas
LADO = 128              # se redimensiona a LADO x LADO
CSV = "rasgos_papa.csv"

In [ ]:
def rasgos_imagen(img, lado=LADO):
    """Resume una hoja en rasgos tabulares de color y textura."""
    a = np.asarray(img.convert("RGB").resize((lado, lado)), dtype=np.float32)
    R, G, B = a[:, :, 0], a[:, :, 1], a[:, :, 2]
    gris = 0.299 * R + 0.587 * G + 0.114 * B
    exg = 2 * G - R - B                              # exceso de verde (hoja sana = mas verde)
    necrosis = float(np.mean((R > G) & (R > B)))     # fraccion parduzca/rojiza (lesiones)
    gx, gy = np.gradient(gris)
    textura = float(np.std(np.sqrt(gx ** 2 + gy ** 2)))
    return [R.mean(), G.mean(), B.mean(), R.std(), G.std(), B.std(),
            float(exg.mean()), necrosis, float(gris.mean()), textura]

COLUMNAS = ["R_media","G_media","B_media","R_std","G_std","B_std",
            "exg_media","necrosis_frac","brillo_media","textura"]

In [ ]:
def construir_tabla():
    filas, y = [], []
    for clase, (ruta, etiqueta) in ZIPS.items():
        if not os.path.exists(ruta):
            print("Falta:", ruta, "- saltando"); continue
        z = zipfile.ZipFile(ruta)
        nombres = [n for n in z.namelist() if n.lower().endswith((".jpg",".jpeg",".png"))]
        random.Random(SEMILLA).shuffle(nombres)
        if POR_CLASE: nombres = nombres[:POR_CLASE]
        for i, n in enumerate(nombres):
            try:
                img = Image.open(io.BytesIO(z.read(n)))
                filas.append(rasgos_imagen(img)); y.append(etiqueta)
            except Exception:
                continue
        print(f"{clase}: {len(nombres)} fotos")
    df = pd.DataFrame(filas, columns=COLUMNAS); df["enferma"] = y
    return df

# Cachea: si ya existe el CSV, lo carga (la extraccion es lo lento)
if os.path.exists(CSV):
    df = pd.read_csv(CSV); print("CSV cargado:", df.shape)
else:
    df = construir_tabla(); df.to_csv(CSV, index=False); print("Tabla construida:", df.shape)
df.head()

## 2. Exploracion minima (EDA) - cada hallazgo decide algo

In [ ]:
print("Balance de clases (enferma=1):")
print(df["enferma"].value_counts(normalize=True).round(3))
print("\nLinea base de clase mayoritaria =", round(df["enferma"].value_counts(normalize=True).max(), 3))
df[COLUMNAS].describe().round(2)

In [ ]:
# Distribucion de dos rasgos clave por clase: verde (exg) y necrosis
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for v, nombre in zip(["exg_media","necrosis_frac"], ax):
    for et, col in [(0,"tab:green"), (1,"tab:brown")]:
        nombre.hist(df[df.enferma==et][v], bins=30, alpha=0.6, color=col,
                    label=("sana" if et==0 else "enferma"))
    nombre.set_title(v); nombre.legend()
plt.tight_layout(); plt.show()

### Decisiones que deja el EDA
- Clases desbalanceadas (mayoria "enferma"): la particion sera **estratificada** por clase, y reportamos metricas por clase, no solo exactitud.
- Las escalas de los rasgos son muy distintas (medias de 0 a 255 frente a fracciones de 0 a 1): hay que **estandarizar**, ajustando el escalador **solo con train** (regla anti-fuga).
- Nota de particion (criterio del profe): lo ideal es partir por grupo (finca o fecha) para evitar casi duplicados. El dataset no trae finca y solo la mitad trae fecha, asi que usamos particion estratificada por clase y lo declaramos como limitacion.

In [ ]:
X = df[COLUMNAS].to_numpy("float32"); y = df["enferma"].to_numpy("float32")
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMILLA)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEMILLA)
escalador = StandardScaler().fit(X_tr)              # SOLO train
X_tr, X_va, X_te = (escalador.transform(a) for a in (X_tr, X_va, X_te))
print("train/val/test:", len(y_tr), len(y_va), len(y_te))

## 3. Arquitectura del MLP
La ultima capa devuelve un **logit** (sin sigmoide en el forward): es la pareja correcta de `BCEWithLogitsLoss`, que combina sigmoide y entropia cruzada en un paso numericamente estable. Poner `nn.Sigmoid()` en el modelo y ademas usar `BCEWithLogitsLoss` seria la **doble sigmoide** (ver Anexo A).

In [ ]:
class CentinelaMLP(nn.Module):
    def __init__(self, d_entrada, oculta1=16, oculta2=8, p_dropout=0.0):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(d_entrada, oculta1), nn.ReLU(), nn.Dropout(p_dropout),  # capa oculta 1
            nn.Linear(oculta1, oculta2),  nn.ReLU(), nn.Dropout(p_dropout),  # capa oculta 2
            nn.Linear(oculta2, 1))                                           # salida: LOGIT
    def forward(self, x):
        return self.red(x)   # la sigmoide vive en la perdida y en la evaluacion

print(CentinelaMLP(len(COLUMNAS)))

## 4. Perdida, optimizador y bucle de entrenamiento explicito

In [ ]:
def a_tensor(Xa, ya):
    return (torch.as_tensor(Xa, dtype=torch.float32),
            torch.as_tensor(ya, dtype=torch.float32).view(-1, 1))

def entrenar(modelo, Xtr, ytr, Xva, yva, epocas=60, lr=1e-2, paciencia=12):
    Xtr, ytr = a_tensor(Xtr, ytr); Xva, yva = a_tensor(Xva, yva)
    criterio = nn.BCEWithLogitsLoss()                      # sigmoide + BCE estable
    optim = torch.optim.Adam(modelo.parameters(), lr=lr)
    hist = {"train": [], "val": []}
    mejor, mejor_ep, espera, mejor_estado = float("inf"), 0, 0, None
    for ep in range(1, epocas + 1):
        modelo.train()                                     # 0 Dropout activo
        optim.zero_grad()                                  # 3 limpiar gradientes
        logits = modelo(Xtr)                               # 1 forward (logit)
        perdida = criterio(logits, ytr)                    # 2 perdida
        perdida.backward()                                 # 4 backward
        optim.step()                                       # 5 actualizar
        modelo.eval()
        with torch.no_grad():
            pv = criterio(modelo(Xva), yva).item()
        hist["train"].append(perdida.item()); hist["val"].append(pv)
        if pv < mejor:
            mejor, mejor_ep, espera = pv, ep, 0
            mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
        else:
            espera += 1
            if paciencia and espera >= paciencia:
                break
    if mejor_estado:
        modelo.load_state_dict(mejor_estado)               # restaura el mejor
    return hist, mejor_ep

## 5. Funciones de diagnostico y linea base

In [ ]:
def plot_curvas(hist, titulo="Curva de perdida"):
    plt.figure(figsize=(6,3.5))
    plt.plot(hist["train"], label="train"); plt.plot(hist["val"], label="val")
    plt.xlabel("epoca"); plt.ylabel("BCE"); plt.title(titulo); plt.legend(); plt.show()

def plot_matriz(y_true, y_pred, etiquetas=("sana","enferma")):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    plt.figure(figsize=(3.6,3.2)); plt.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i,j], ha="center", va="center")
    plt.xticks([0,1], etiquetas); plt.yticks([0,1], etiquetas)
    plt.xlabel("prediccion"); plt.ylabel("real"); plt.title("Matriz de confusion"); plt.show()

def probas(modelo, Xa):
    modelo.eval()
    with torch.no_grad():
        return torch.sigmoid(modelo(torch.as_tensor(Xa, dtype=torch.float32))).numpy().ravel()

# Linea base comprometida: clase mayoritaria y un rasgo crudo (necrosis) con umbral en val
base_may = max(y_te.mean(), 1 - y_te.mean())
idx_necro = COLUMNAS.index("necrosis_frac")
# el escalador cambio la escala; usamos el rasgo crudo desde df para la base
print("Linea base clase mayoritaria (exactitud):", round(float(base_may), 3))

## 6. Desafio - inducir y corregir el sobreajuste con Dropout
Dos corridas comparables con la misma semilla: una red grande sin Dropout (tiende a sobreajustar) y la misma con Dropout. Se nombra la epoca del minimo de validacion de cada una.

In [ ]:
torch.manual_seed(SEMILLA)
m_grande = CentinelaMLP(len(COLUMNAS), 128, 64, p_dropout=0.0)
h_grande, ep_g = entrenar(m_grande, X_tr, y_tr, X_va, y_va, epocas=120, lr=1e-2, paciencia=None)

torch.manual_seed(SEMILLA)
m_drop = CentinelaMLP(len(COLUMNAS), 128, 64, p_dropout=0.3)
h_drop, ep_d = entrenar(m_drop, X_tr, y_tr, X_va, y_va, epocas=120, lr=1e-2, paciencia=None)

plot_curvas(h_grande, f"Sin Dropout (min val en epoca {ep_g}, val={min(h_grande['val']):.3f})")
plot_curvas(h_drop,   f"Con Dropout 0.3 (min val en epoca {ep_d}, val={min(h_drop['val']):.3f})")
print("Brecha train-val sin dropout:", round(h_grande['val'][ep_g-1]-h_grande['train'][ep_g-1], 3))
print("Brecha train-val con dropout:", round(h_drop['val'][ep_d-1]-h_drop['train'][ep_d-1], 3))

## 7. Umbral en validacion, evaluacion en prueba, errores y etica
El umbral se elige en **validacion** segun el costo de cada error (nunca en prueba). Para un centinela, un falso negativo -no avisar un brote- suele ser mas costoso que una falsa alarma, asi que conviene un umbral que favorezca el recall de "enferma".

In [ ]:
modelo = m_drop   # modelo final: el regularizado
prob_va = probas(modelo, X_va); prob_te = probas(modelo, X_te)
print("ROC-AUC validacion:", round(roc_auc_score(y_va, prob_va), 3))

# Elegir umbral en validacion maximizando F1 de la clase de riesgo
prec, rec, umbrales = precision_recall_curve(y_va, prob_va)
f1 = 2*prec*rec/(prec+rec+1e-9)
umbral = float(umbrales[np.argmax(f1[:-1])])
print("Umbral elegido en validacion:", round(umbral, 3))

pred_te = (prob_te >= umbral).astype(int)
print("\nROC-AUC prueba:", round(roc_auc_score(y_te, prob_te), 3))
print("Linea base (clase mayoritaria):", round(float(base_may), 3))
plot_matriz(y_te, pred_te)
print(classification_report(y_te, pred_te, target_names=["sana","enferma"], digits=3))

### Lectura tecnica y etica de los errores
- Comparar el ROC-AUC y el recall de "enferma" contra la linea base de clase mayoritaria: el MLP debe aportar algo mas que decir "enferma" a todo.
- Falsos negativos (hoja enferma marcada como sana): dejan avanzar el brote. Falsos positivos: fumigacion innecesaria. Para el cliente (extension agricola) el falso negativo suele ser mas caro.
- Marco etico para la IA en Colombia: una alerta temprana agricola debe ser transparente sobre su alcance (modelo entrenado en una variedad y region de Tanzania) y no reemplazar el criterio del tecnico.

## 8. Cierre - autoevaluacion contra la rubrica
Marca honestamente el nivel por criterio: Propuesta+EDA (15%), MLP (30%), Regularizacion (15%), Evaluacion+etica (15%), uso de IA (15%), comunicacion (10%). Aprueba con >= 3.0 ponderado.

## Anexo A - Auditoria a la IA: la doble sigmoide, reproducida
Un asistente propuso poner `nn.Sigmoid()` en el modelo y ademas usar `BCEWithLogitsLoss`. El codigo corre, pero aplica la sigmoide dos veces: la perdida se estanca y las probabilidades se comprimen. Abajo se reproduce el fallo y se contrasta con la version correcta.

In [ ]:
class MLPDobleSigmoide(nn.Module):
    """Version rota: sigmoide en el forward + BCEWithLogitsLoss (aplica sigmoide dos veces)."""
    def __init__(self, d):
        super().__init__()
        self.red = nn.Sequential(nn.Linear(d,16), nn.ReLU(),
                                 nn.Linear(16,8), nn.ReLU(),
                                 nn.Linear(8,1), nn.Sigmoid())   # <- sigmoide de mas
    def forward(self, x): return self.red(x)

torch.manual_seed(SEMILLA)
m_rota = MLPDobleSigmoide(len(COLUMNAS))
h_rota, _ = entrenar(m_rota, X_tr, y_tr, X_va, y_va, epocas=60, lr=1e-2, paciencia=None)
print("Perdida val minima ROTA   (doble sigmoide):", round(min(h_rota['val']), 3), "-> se estanca alto")
print("Perdida val minima CORRECTA (logit):        ", round(min(h_drop['val']), 3))
print("AUC prueba rota:", round(roc_auc_score(y_te, probas(m_rota, X_te)), 3),
      "| correcta:", round(roc_auc_score(y_te, prob_te), 3))

## Anexo B - Propuesta de Datos
La Propuesta de Datos aprobada (compuerta, Paso 0) esta en el repositorio: `propuesta/propuesta-de-datos.md`. Fuentes: Irish Potato Imagery Dataset (Zenodo, CC BY 4.0) y NASA POWER.

## Anexo C - Bitacora de IA (plantilla)
| Fecha | Sugerencia de la IA | Decision (Aceptada/Modificada/Rechazada) | Justificacion y evidencia |
|---|---|---|---|
| 2026-11-02 | poner Sigmoid en el forward con BCEWithLogitsLoss | Rechazada | doble sigmoide; se reproduce el fallo en el Anexo A y se corrige con salida en logit |
| | | | |
| | | | |

## Anexo D - Informe tecnico-etico (<= 4 paginas)
Cinco secciones: 1) problema y cliente; 2) datos; 3) modelo y entrenamiento; 4) resultados y errores (FN y FP); 5) decision recomendada y riesgos eticos. Referencias en APA 7. Se entrega como PDF aparte.